## Imports

In [1]:
# Cell 1: Imports
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks


##  Load data

In [2]:
# Cell 2: Load data
DATA_PATH = "../data/Malware dataset.csv"   # <-- change
TARGET_COL = "label"               # <-- change (e.g., 'class', 'Label', 'target')

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print(df['classification'].value_counts())


Shape: (100000, 35)


,hash,millisecond,classification,state,usage_counter,prio,static_prio,normal_prio,policy,vm_pgoff,...,nivcsw,min_flt,maj_flt,fs_excl_counter,lock,utime,stime,gtime,cgtime,signal_nvcsw
0,42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...,0,malware,0,0,3069378560,14274,0,0,0,...,0,0,120,0,3204448256,380690,4,0,0,0
1,42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...,1,malware,0,0,3069378560,14274,0,0,0,...,0,0,120,0,3204448256,380690,4,0,0,0
2,42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...,2,malware,0,0,3069378560,14274,0,0,0,...,0,0,120,0,3204448256,380690,4,0,0,0
3,42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...,3,malware,0,0,3069378560,14274,0,0,0,...,0,0,120,0,3204448256,380690,4,0,0,0
4,42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...,4,malware,0,0,3069378560,14274,0,0,0,...,0,0,120,0,3204448256,380690,4,0,0,0


classification
malware    50000
benign     50000
Name: count, dtype: int64


## Basic cleaning

In [3]:
# Cell 3: Basic cleaning
df = df.drop_duplicates()
df = df.fillna(0)

X = df.drop(columns=['classification','hash'])
y = df['classification']

# Encode labels if not numeric
if y.dtype == "object":
    le = LabelEncoder()
    y = le.fit_transform(y)
    print("Classes:", list(le.classes_))
else:
    le = None


Classes: ['benign', 'malware']


## Train-test split + scaling

In [4]:
# Cell 4: Train-test split + scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape, "Test:", X_test_scaled.shape)


Train: (80000, 33) Test: (20000, 33)


In [5]:
len(np.unique(y_train))

2

## Build ANN model

In [6]:
# ==============================
# ANN Model for Binary Classification
# ==============================

input_dim = X_train_scaled.shape[1]

ann_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(input_dim,)),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),

    tf.keras.layers.Dense(1, activation='sigmoid')  # Binary output
])

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │           2,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,801 (18.75 KB)

 Trainable params: 4,801 (18.75 KB)

 Non-trainable params: 0 (0.00 B)

##  Build DNN model

In [7]:
# Cell: Build Binary DNN (Fix for your error)
input_dim = X_train_scaled.shape[1]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(input_dim,)),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(1, activation="sigmoid")  # ✅ must be 1 for binary
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                      │ (None, 256)                 │           8,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 51,457 (201.00 KB)

 Trainable params: 50,689 (198.00 KB)

 Non-trainable params: 768 (3.00 KB)

## Train

In [ ]:
# Cell 6: Train
cbs = [
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint("malware_dnn.h5", monitor="val_loss", save_best_only=True)
]

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=cbs,
    verbose=1
)


Epoch 1/30
 991/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9352 - loss: 0.1514

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9703 - loss: 0.0749 - val_accuracy: 0.9980 - val_loss: 0.0072
Epoch 2/30
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9910 - loss: 0.0235

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9926 - loss: 0.0202 - val_accuracy: 0.9984 - val_loss: 0.0040
Epoch 3/30
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9950 - loss: 0.0141

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9956 - loss: 0.0130 - val_accuracy: 0.9996 - val_loss: 0.0013
Epoch 4/30
 998/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9972 - loss: 0.0080

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9970 - loss: 0.0084 - val_accuracy: 0.9998 - val_loss: 7.0110e-04
Epoch 5/30
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.9972 - loss: 0.0087 - val_accuracy: 0.9999 - val_loss: 9.7145e-04
Epoch 6/30
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9978 - loss: 0.0060 - val_accuracy: 0.9996 - val_loss: 0.0011
Epoch 7/30
 998/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9983 - loss: 0.0057

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.9980 - loss: 0.0068 - val_accuracy: 0.9999 - val_loss: 3.1167e-04
Epoch 8/30
 858/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9974 - loss: 0.0074

## Plot training curves

In [ ]:
# Cell 7: Plot training curves
plt.figure()
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure()
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
# Cell 8: Predictions
y_pred_raw = model.predict(X_test_scaled)

if num_classes > 2:
    y_pred = np.argmax(y_pred_raw, axis=1)
else:
    y_pred = (y_pred_raw.reshape(-1) >= 0.5).astype(int)

print("Sample preds:", y_pred[:20])


## Classification Report

In [ ]:
# Cell 9: Classification Report (IMPORTANT)
print("=== Classification Report ===")
if le is not None:
    # Build readable target names
    target_names = [str(c) for c in le.classes_]
    print(classification_report(y_test, y_pred, target_names=target_names))
else:
    print(classification_report(y_test, y_pred))


## Confusion Matrix

In [ ]:
# Cell 10: Confusion Matrix (IMPORTANT)
cm = confusion_matrix(y_test, y_pred)

plt.figure()
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("Actual")

# Write values on cells
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j],
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.show()


## Scaler + label encoder + feature columns + trained DNN

In [ ]:
# Cell: Save preprocessing + model
import joblib
import json

# 1) Save scaler
joblib.dump(scaler, "../models/scaler.pkl")

# 2) Save label encoder if you used it
if le is not None:
    joblib.dump(le, "../models/label_encoder.pkl")
else:
    # optional: save a dummy mapping for binary labels
    with open("label_mapping.json", "w") as f:
        json.dump({"0": "Benign", "1": "Malware"}, f)

# 3) Save feature column order (VERY IMPORTANT for future prediction)
feature_cols = list(X.columns)
joblib.dump(feature_cols, "../models/feature_cols.pkl")

# 4) Save the DNN model
model.save("../models/malware_dnn.h5")   # or model.save("malware_dnn_savedmodel")
print("✅ Saved: scaler.pkl, (label_encoder.pkl), feature_cols.pkl, malware_dnn.h5")
